In [1]:
import torch
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
import cv2
import os
import math

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"use {device}")
dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16

model = VGGT.from_pretrained("facebook/VGGT-1B").to(device)
print("VGGT-1B loaded")

use cuda


'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /facebook/VGGT-1B/resolve/main/config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x00000133166EB2B0>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: 8fec0156-1f42-4d23-bd64-eae409c35dbf)')' thrown while requesting HEAD https://huggingface.co/facebook/VGGT-1B/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError("HTTPSConnectionPool(host='huggingface.co', port=443): Max retries exceeded with url: /facebook/VGGT-1B/resolve/main/config.json (Caused by ConnectTimeoutError(<urllib3.connection.HTTPSConnection object at 0x00000133166EB670>, 'Connection to huggingface.co timed out. (connect timeout=10)'))"), '(Request ID: c05444fb-7c09-4e0b-bfce-be05e67d2bbc)')' thrown while requesting HEAD https://huggingface.co/facebook/VGGT-1B/resolve/main/config.json
Retrying in 2s [Retry 2/5].


VGGT-1B loaded


In [ ]:
input_video = "Saure.mp4"
output_dir = "frame"
os.makedirs(output_dir, exist_ok=True)

cap = cv2.VideoCapture(input_video)
frame_interval = 10
frame_count = 0
saved_count = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    if frame_count % frame_interval == 0:
        output_path = os.path.join(output_dir, f"{saved_count:04d}.png")
        cv2.imwrite(output_path, frame)
        saved_count += 1
    frame_count += 1
print(f"{saved_count} frames saved")
cap.release()

In [3]:
# ============== 核心：提取目录下所有.jpg图片 ==============
# 定义目标目录路径（与脚本同级目录下的frames文件夹）
frame_dir = r"frames"

# 检查目录是否存在
if not os.path.isdir(frame_dir):
    raise FileNotFoundError(f"{frame_dir} does not exist")

valid_files = []
for filename in os.listdir(frame_dir):
    file_path = os.path.join(frame_dir, filename)
    if os.path.isfile(file_path):
        valid_files.append(filename)

valid_files.sort()

image_names = [os.path.join(frame_dir, filename) for filename in valid_files]
print(f"{len(image_names)} images found")

if not image_names:
    raise ValueError(f"no images found in {frame_dir}")

10 images found


> 12 images for each batch

In [4]:
batch_count = 2
batch_size = math.ceil(len(image_names) / 1)  # 向上取整，确保均分
all_wp = []
all_wp_conf = []
all_imgs = []
batch_counter = 0
for i in range(1):
    batch_names = image_names
    batch_images = load_and_preprocess_images(batch_names).to(device)
    
    with torch.no_grad():
        with torch.cuda.amp.autocast(dtype=dtype):
            predictions = model(batch_images)
    
    # print(predictions.keys())
    wp = predictions["world_points"].detach().float()
    wp_conf = predictions["world_points_conf"].detach().float()
    imgs = batch_images.detach().float().clamp(0, 1)
    
    all_wp.append(wp)
    all_wp_conf.append(wp_conf)
    all_imgs.append(imgs)
    
    # 释放当前批次的显存
    del batch_images, predictions
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()  # 额外回收跨进程的未使用显存

    print(f"Processed batch {i+1}/{batch_count}")

# 合并所有批次的结果
wp = torch.cat(all_wp, dim=1)  # 按时间维度（T）拼接
wp_conf = torch.cat(all_wp_conf, dim=1)
imgs = torch.cat(all_imgs, dim=0)  # 按批次维度拼接

e:\vggt\vggt\layers\attention.py:61: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:455.)
  x = F.scaled_dot_product_attention(q, k, v, dropout_p=self.attn_drop.p if self.training else 0.0)


Processed batch 1/2


In [ ]:
# # 滑动窗口参数
# window_size = 5  # 每次推理5张图片
# n_images = len(image_names)  # 总图片数量

# # 计算滑动窗口数量：当图片数>=5时，窗口数为n-4；不足5张时只处理1次
# batch_count = max(1, n_images - window_size + 1) if n_images >= window_size else 1

# all_wp = []
# all_wp_conf = []
# all_imgs = []

# for i in range(batch_count):
#     # 计算当前窗口的起止索引（处理最后一个窗口可能不足5张的情况）
#     start_idx = i
#     end_idx = start_idx + window_size
#     end_idx = min(end_idx, n_images)  # 避免索引超出范围
    
#     # 获取当前窗口的图片名称
#     batch_names = image_names[start_idx:end_idx]
#     print(f"Processing window {i+1}/{batch_count}: images {start_idx} to {end_idx-1}")
    
#     # 加载并预处理当前窗口的图片
#     batch_images = load_and_preprocess_images(batch_names).to(device)
    
#     # 推理
#     with torch.no_grad():
#         with torch.cuda.amp.autocast(dtype=dtype):
#             predictions = model(batch_images)
    
#     # 提取结果
#     wp = predictions["world_points"].detach().float()
#     wp_conf = predictions["world_points_conf"].detach().float()
#     imgs = batch_images.detach().float().clamp(0, 1)
    
#     # 收集结果
#     all_wp.append(wp)
#     all_wp_conf.append(wp_conf)
#     all_imgs.append(imgs)
    
#     # 释放显存
#     del batch_images, predictions
#     torch.cuda.empty_cache()
#     torch.cuda.ipc_collect()

# # 合并所有窗口的结果（根据实际维度调整拼接方向）
# wp = torch.cat(all_wp, dim=1)
# wp_conf = torch.cat(all_wp_conf, dim=1)
# imgs = torch.cat(all_imgs, dim=0)

# print(f"All windows processed. Total results shape - wp: {wp.shape}, imgs: {imgs.shape}")

Processing window 1/21: images 0 to 4
Processing window 2/21: images 1 to 5
Processing window 3/21: images 2 to 6
Processing window 4/21: images 3 to 7
Processing window 5/21: images 4 to 8
Processing window 6/21: images 5 to 9
Processing window 7/21: images 6 to 10
Processing window 8/21: images 7 to 11
Processing window 9/21: images 8 to 12
Processing window 10/21: images 9 to 13
Processing window 11/21: images 10 to 14
Processing window 12/21: images 11 to 15
Processing window 13/21: images 12 to 16
Processing window 14/21: images 13 to 17
Processing window 15/21: images 14 to 18
Processing window 16/21: images 15 to 19
Processing window 17/21: images 16 to 20
Processing window 18/21: images 17 to 21
Processing window 19/21: images 18 to 22
Processing window 20/21: images 19 to 23
Processing window 21/21: images 20 to 24
All windows processed. Total results shape - wp: torch.Size([1, 105, 392, 518, 3]), imgs: torch.Size([105, 3, 392, 518])


In [5]:
# ============== Export concise colored point clouds (per-frame) ==============
def save_ply(xyz, rgb, out_path):
    n = xyz.shape[0]
    with open(out_path, "w") as f:
        f.write("ply\n")
        f.write("format ascii 1.0\n")
        f.write(f"element vertex {n}\n")
        f.write("property float x\n")
        f.write("property float y\n")
        f.write("property float z\n")
        if rgb is not None:
            f.write("property uchar red\n")
            f.write("property uchar green\n")
            f.write("property uchar blue\n")
        f.write("end_header\n")
        if rgb is not None:
            for (x, y, z), (r, g, b) in zip(xyz.tolist(), rgb.tolist()):
                f.write(f"{x} {-y} {-z} {int(r)} {int(g)} {int(b)}\n")
        else:
            for (x, y, z) in xyz.tolist():
                f.write(f"{x} {-y} {-z}\n")

In [6]:
_, T, H, W, _ = wp.shape
os.makedirs("outputs", exist_ok=True)

# ============== Fuse all frames into a single scene point cloud ==============
# 按每帧各自中位数阈值筛选，再合并
all_xyz = []
all_rgb = []
for t in range(T):
    P = wp[0, t]                 # [H, W, 3]
    C = wp_conf[0, t]            # [H, W]
    I = imgs[t]                  # [3, H, W]

    xyz = P.reshape(-1, 3)
    conf = C.reshape(-1)
    thr = float(conf.median())
    mask = conf > thr

    all_xyz.append(xyz[mask])
    all_rgb.append((I.reshape(3, -1).T[mask] * 255.0).to(torch.uint8))

scene_xyz = torch.cat(all_xyz, dim=0)
scene_rgb = torch.cat(all_rgb, dim=0)
scene_out = os.path.join("outputs", "scene.ply")
save_ply(scene_xyz, scene_rgb, scene_out)
print(f"Saved fused scene: {scene_out} | points={scene_xyz.shape[0]}")

Saved fused scene: outputs\scene.ply | points=761459
